In [17]:
%iam_role arn:aws:iam::770170581396:role/aws-glue-s3-permission
%region eu-north-1
%idle_timeout 15
%worker_type G.1X
%number_of_workers 2
%glue_version 5.0

Current iam_role is arn:aws:iam::770170581396:role/aws-glue-s3-permission
iam_role has been set to arn:aws:iam::770170581396:role/aws-glue-s3-permission.
Previous region: eu-north-1
Setting new region to: eu-north-1
Region is set to: eu-north-1
Current idle_timeout is 15 minutes.
idle_timeout has been set to 15 minutes.
Previous worker type: G.1X
Setting new worker type to: G.1X
Previous number of workers: 2
Setting new number of workers to: 2
Setting Glue version to: 5.0


# Bronze to Silver

Analyze the lossless Bronze Parquet data, then safely type, validate, clean, enrich, and product-deduplicate it. Punctuation and sentence boundaries are retained for future aspect-based sentiment analysis.

In [1]:
from pyspark.sql import Window
from pyspark.sql import functions as F

BRONZE_PATH = "s3://amazon-food-reviews-ml-model/bronze/"
SILVER_PATH = "s3://amazon-food-reviews-ml-model/silver/"

bronze_df = spark.read.parquet(BRONZE_PATH).cache()
bronze_count = bronze_df.count()
if bronze_count == 0:
    raise ValueError(f"No Bronze data found at {BRONZE_PATH}")

print(f"Bronze rows: {bronze_count:,}")
bronze_df.printSchema()
bronze_df.show(5, truncate=80)

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 15
Session ID: a24def13-1b8b-448f-9037-20b4ac1a4874
Applying the following default arguments:
--glue_kernel_version 1.0.9
--enable-glue-datacatalog true
Waiting for session a24def13-1b8b-448f-9037-20b4ac1a4874 to get into ready status...
Session a24def13-1b8b-448f-9037-20b4ac1a4874 has been created.
Bronze rows: 568,454
root
 |-- raw_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- profile_name: string (nullable = true)
 |-- raw_helpfulness_numerator: string (nullable = true)
 |-- raw_helpfulness_denominator: string (nullable = true)
 |-- raw_score: string (nullable = true)
 |-- raw_review_time: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _r

## 1. Bronze Analysis

In [2]:
required_columns = {
    "raw_id", "product_id", "user_id", "profile_name",
    "raw_helpfulness_numerator", "raw_helpfulness_denominator",
    "raw_score", "raw_review_time", "summary", "review_text",
    "_source_file", "_record_id", "_bronze_ingested_at",
}
missing_columns = sorted(required_columns - set(bronze_df.columns))
if missing_columns:
    raise ValueError(f"Missing Bronze columns: {missing_columns}")

print("Null or blank source fields:")
bronze_df.select([
    F.count(F.when(
        F.col(column).isNull()
        | (F.trim(F.col(column).cast("string")) == ""), 1
    )).alias(column)
    for column in [
        "raw_id", "product_id", "user_id", "raw_score",
        "raw_review_time", "summary", "review_text",
    ]
]).show(truncate=False)

print("Raw rating distribution:")
bronze_df.groupBy("raw_score").count().orderBy("raw_score").show(20)

duplicate_record_ids = bronze_count - bronze_df.select("_record_id").distinct().count()
print(f"Duplicate Bronze record IDs: {duplicate_record_ids:,}")

Null or blank source fields:
+------+----------+-------+---------+---------------+-------+-----------+
|raw_id|product_id|user_id|raw_score|raw_review_time|summary|review_text|
+------+----------+-------+---------+---------------+-------+-----------+
|0     |0         |0      |0        |0              |0      |0          |
+------+----------+-------+---------+---------------+-------+-----------+

Raw rating distribution:
+---------+------+
|raw_score| count|
+---------+------+
|        1| 52268|
|        2| 29769|
|        3| 42640|
|        4| 80655|
|        5|363122|
+---------+------+

Duplicate Bronze record IDs: 0


In [3]:
integer_pattern = r"^[+-]?[0-9]+$"
parseability_df = bronze_df.select(
    F.count(F.when(~F.trim(F.col("raw_id")).rlike(integer_pattern), 1)).alias("invalid_id"),
    F.count(F.when(~F.trim(F.col("raw_score")).rlike(integer_pattern), 1)).alias("invalid_score"),
    F.count(F.when(~F.trim(F.col("raw_review_time")).rlike(integer_pattern), 1)).alias("invalid_time"),
    F.count(F.when(~F.trim(F.col("raw_helpfulness_numerator")).rlike(integer_pattern), 1)).alias("invalid_helpfulness_numerator"),
    F.count(F.when(~F.trim(F.col("raw_helpfulness_denominator")).rlike(integer_pattern), 1)).alias("invalid_helpfulness_denominator"),
)
parseability_df.show(truncate=False)

+----------+-------------+------------+-----------------------------+-------------------------------+
|invalid_id|invalid_score|invalid_time|invalid_helpfulness_numerator|invalid_helpfulness_denominator|
+----------+-------------+------------+-----------------------------+-------------------------------+
|0         |0            |0           |0                            |0                              |
+----------+-------------+------------+-----------------------------+-------------------------------+


## 2. Silver Transformations

Scores 1–2 become negative, 3 remains neutral, and 4–5 become positive. The binary label is null for neutral reviews so they remain available for future aspect analysis without contaminating binary model training.

In [4]:
def safe_integral(column_name, data_type):
    value = F.trim(F.col(column_name))
    return F.when(value.rlike(r"^[+-]?[0-9]+$"), value.cast(data_type))

review_time_epoch = safe_integral("raw_review_time", "long")

typed_df = bronze_df.select(
    safe_integral("raw_id", "long").alias("id"),
    F.trim("product_id").alias("product_id"),
    F.trim("user_id").alias("user_id"),
    F.trim("profile_name").alias("profile_name"),
    safe_integral("raw_helpfulness_numerator", "int").alias("helpfulness_numerator"),
    safe_integral("raw_helpfulness_denominator", "int").alias("helpfulness_denominator"),
    safe_integral("raw_score", "int").alias("score"),
    F.col("raw_review_time").alias("review_time_raw"),
    review_time_epoch.alias("review_time_epoch"),
    F.from_unixtime(review_time_epoch).cast("timestamp").alias("review_timestamp"),
    "summary", "review_text", "_source_file", "_record_id", "_bronze_ingested_at",
)

cleaned_df = (
    typed_df
    .withColumn("summary", F.trim("summary"))
    .withColumn("review_text_clean", F.col("review_text"))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)<br\s*/?>", ". "))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)</?p\s*>", ". "))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)<[^>]+>", " "))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)&amp;", " and "))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)&quot;", '"'))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)&(?:apos|#39);", "'"))
    .withColumn("review_text_clean", F.regexp_replace("review_text_clean", r"(?i)&(?:nbsp|lt|gt);|&#[0-9]+;", " "))
    .withColumn("review_text_clean", F.trim(F.regexp_replace("review_text_clean", r"\s+", " ")))
    .withColumn("normalized_text", F.trim(F.regexp_replace(F.lower("review_text_clean"), r"\s+", " ")))
)

In [5]:
valid_df = (
    cleaned_df
    .filter(F.col("id").isNotNull())
    .filter(F.length("product_id") > 0)
    .filter(F.length("user_id") > 0)
    .filter(F.col("score").between(1, 5))
    .filter(F.length("review_text_clean") > 0)
    .filter(F.col("helpfulness_numerator") >= 0)
    .filter(F.col("helpfulness_denominator") >= 0)
    .filter(F.col("helpfulness_numerator") <= F.col("helpfulness_denominator"))
)

dedupe_window = Window.partitionBy(
    "product_id", "user_id", "review_time_epoch", "normalized_text"
).orderBy(F.col("id"), F.col("_record_id"))

silver_df = (
    valid_df
    .withColumn("_duplicate_rank", F.row_number().over(dedupe_window))
    .filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank")
    .withColumn("sentiment_class",
        F.when(F.col("score") <= 2, "negative")
         .when(F.col("score") == 3, "neutral").otherwise("positive"))
    .withColumn("binary_label",
        F.when(F.col("score") <= 2, 0)
         .when(F.col("score") >= 4, 1).otherwise(F.lit(None).cast("int")))
    .withColumn("review_length", F.length("review_text_clean"))
    .withColumn("review_word_count", F.size(F.split("normalized_text", r"\s+")))
    .withColumn("helpfulness_ratio",
        F.when(F.col("helpfulness_denominator") > 0,
            F.col("helpfulness_numerator") / F.col("helpfulness_denominator")))
    .withColumn("has_valid_review_time", F.col("review_time_epoch").isNotNull())
    .withColumn("_silver_processed_at", F.current_timestamp())
)

silver_count = silver_df.count()
print(f"Silver rows: {silver_count:,}")
silver_df.groupBy("score", "sentiment_class", "binary_label").count().orderBy("score").show()
silver_df.select("review_text", "review_text_clean", "sentiment_class").show(5, truncate=100)

Silver rows: 567,129
+-----+---------------+------------+------+
|score|sentiment_class|binary_label| count|
+-----+---------------+------------+------+
|    1|       negative|           0| 51960|
|    2|       negative|           0| 29751|
|    3|        neutral|        NULL| 42552|
|    4|       positive|           1| 80538|
|    5|       positive|           1|362328|
+-----+---------------+------------+------+

+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+---------------+
|                                                                                         review_text|                                                                                   review_text_clean|sentiment_class|
+----------------------------------------------------------------------------------------------------+----------------------------------------

## 3. Save Silver as Parquet

In [6]:
if silver_count == 0:
    raise ValueError("Silver transformation produced zero rows")

silver_df.write.mode("overwrite").option("compression", "snappy").parquet(SILVER_PATH)
written_count = spark.read.parquet(SILVER_PATH).count()
if written_count != silver_count:
    raise ValueError(f"Silver write failed: expected {silver_count}, got {written_count}")

print(f"Saved {written_count:,} Silver rows to {SILVER_PATH}")
bronze_df.unpersist()

Saved 567,129 Silver rows to s3://amazon-food-reviews-ml-model/silver/
DataFrame[raw_id: string, product_id: string, user_id: string, profile_name: string, raw_helpfulness_numerator: string, raw_helpfulness_denominator: string, raw_score: string, raw_review_time: string, summary: string, review_text: string, _source_file: string, _bronze_ingested_at: timestamp, _record_id: string]


In [19]:
%stop_session

Stopping session: a24def13-1b8b-448f-9037-20b4ac1a4874
Stopped session.
